# Data Cleaning: Impacts of Data Centers on Electrical Utilities

This notebook loads the raw data from `data/raw/`.

In [65]:
import re
import pandas as pd
import numpy as np
from pathlib import Path

In [66]:
DATA_DIR = Path("data/raw")

# display options for pandas
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

## First look at each dataset

### FracTracker U.S. Data Centers

In [67]:
data_centers = pd.read_csv(DATA_DIR / "fractracker_data_centers.csv")
print("Rows and columns:", data_centers.shape)
data_centers.sample(3)

Rows and columns: (1701, 53)


,OBJECTID,GlobalID,facility_id,facility_name,address,city,state,zip,county,lat,long,status,location_confidence,location_determination,purpose,operator_name,tenant,mw,mw_low,mw_high,sizerank,sizerank_numeric,power_source,dedicated_power_plant,number_of_generators,number_of_buildings,cooling_source,cooling_type,facility_size_sqft,property_size_acres,project_cost,expected_date_online,community_pushback,advocacy_information,resistance_status,nda,community_group_website_1,community_group_website_2,petition_url,other_info,editor,information_source,info_source_1,info_source_2,info_source_3,info_source_4,info_source_5,info_source_6,info_source_7,info_source_8,date_created,date_updated,delete_dup
38,3938198,635c339e-faa0-4abc-917e-ad5320304d2c,19,NTT Data Center,22000-21794 E Pecos Rd,Mesa,AZ,85212.0,Maricopa,33.295560,-111.614000,Approved/Permitted/Under construction,High,Google Maps and other sources,NaN,NTT Global Data Centers,NaN,200,200.0,200.0,Hyperscale (100-999 MW),5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,173.0,$300 million,NaN,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KE/MG,Media Monitoring,https://www.datacenterdynamics.com/en/news/ntt...,https://www.datacenterdynamics.com/en/news/two...,NaN,NaN,NaN,NaN,NaN,NaN,1.748390e+12,1.771459e+12,NaN
955,3939115,64b7d40f-fe31-43d5-9c2b-d43f5b57968c,1768501881634-3585,CleanSpark Sealy Data Center,3 miles outside of Sealy,Austin County,TX,77474.0,Austin,29.884085,-96.219376,Proposed,Low,Google Maps and other sources,NaN,CleanSpark,NaN,285,285.0,285.0,Hyperscale (100-999 MW),5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,271.0,NaN,2027,Yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KE/KJ,Media Monitoring,https://www.datacenterdynamics.com/en/news/cle...,https://www.linkedin.com/posts/garyveselka_ai-...,https://sealyforward.com/,https://www.click2houston.com/news/local/2026/...,NaN,NaN,NaN,NaN,1.768435e+12,1.787702e+12,NaN
837,3938997,30e1070b-5b4b-4aa8-9c7e-69ad84049a26,1790126447738-8807,Misty Hollow Energy and Data Campus,614 Prospect Rd,Evans City,PA,16033.0,Butler,40.808056,-80.055840,Proposed,Medium,Google Maps and other sources,NaN,NaN,NaN,NaN,NaN,NaN,Hyperscale (100-999 MW),5,NaN,NaN,NaN,NaN,NaN,Closed loop,1000000.0,274.0,NaN,2029,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KE,Crowdsourced,https://www.cranberryeagle.com/2026/09/18/mass...,https://www.wtae.com/article/pennsylvania-data...,NaN,NaN,NaN,NaN,NaN,NaN,1.790035e+12,1.790035e+12,NaN


In [68]:
# Summary Statistics for Data Centers Dataset
# Looking for anything that needs to be cleaned
data_centers.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
OBJECTID,1701.0,NaN,NaN,NaN,3939010.0,491.18072,3938160.0,3938585.0,3939010.0,3939435.0,3939860.0
GlobalID,1701,1701,7a8ad31d-534e-46fa-9261-36ee41ba940a,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
facility_id,1701,1701,1779305646902-5353,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
facility_name,1701,1567,H5 Data Center,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
address,1656,1644,Farm-to-Market 157,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
city,1697,836,Sterling,82,NaN,NaN,NaN,NaN,NaN,NaN,NaN
state,1701,49,VA,464,NaN,NaN,NaN,NaN,NaN,NaN,NaN
zip,1689.0,NaN,NaN,NaN,43813.92836,26462.614683,1085.0,20166.0,30339.0,72376.0,99734.0
county,1698,478,Loudoun,192,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lat,1701.0,NaN,NaN,NaN,37.620852,4.22314,21.480452,33.8084,38.748386,40.06241,70.18478


### FracTracker U.S. Data Centers - Data Cleaning

| Feature | Identified Issue | Proposed Solution |
|---|---|---|
| `mw` | Contains both numerical values and ranges. | Separate ranges into `mw_low` and `mw_high` to preserve the available information. |
| `power_source` | String categorical values. | Standardize categories and encode them for modeling. |
| `dedicated_power_plant` | Limited availability across facilities. | Determine if power plants can be identified using location (latitude/longitude) and power source. |
| `number_of_generators` | Contains numerical information stored as strings; limited availability across facilities. | Too limited to be a learnable feature unless more information can be added. |
| `number_of_buildings` | Contains integers, ranges, and strings (e.g., `6+`, `1-19`, `8`, `3 large, 1 small`). | Create a consistent approach for representing ranges and descriptive counts. |
| `cooling_source` | Categorical values stored as strings. | Standardize categories and encode them for modeling. |
| `cooling_type` | Inconsistent categorical formatting (e.g., `Closed loop` vs. `Closed Loop`). | Normalize categorical values and encode them for modeling. |
| `project_cost` | Contains strings and potentially ranges rather than consistent numerical values. | Extract numerical values and determine an appropriate method for representing ranges. |



`location_confidence`

In [69]:
# Small fix for location confidence values
data_centers['location_confidence'] = (
    data_centers['location_confidence']
    .apply(lambda x: 'High' if x == 'high' else x)
)

`mw`

In [70]:
# Remove commas and spaces
mw = data_centers['mw'].str.replace(',', '').str.strip()

# Split and assign ranges
data_centers[['mw_low', 'mw_high']] = mw.str.split('-', expand=True)
data_centers['mw_high'] = data_centers['mw_high'].fillna(data_centers['mw_low']) # fill where no range exists 

# Convert to numeric
data_centers[['mw_low', 'mw_high']] = (
    data_centers[['mw_low', 'mw_high']]
    .apply(pd.to_numeric, errors='coerce')
)

data_centers.loc[data_centers['mw'].notna()][['mw', 'mw_low', 'mw_high']].head()

,mw,mw_low,mw_high
0,3000,3000.0,3000.0
1,120,120.0,120.0
2,1200,1200.0,1200.0
3,300,300.0,300.0
6,210,210.0,210.0


`power_source`

In [71]:
# Normalize power sources
power_source = (
    data_centers['power_source']
    .str.lower()
    .str.strip()
    .str.replace('(', '')
    .str.replace(')', '')
    .str.replace('/', '_')
    .str.replace(' ', '_')
)

# One-hot encoding for each source type
source_encoded = power_source.str.get_dummies(sep=",_")
data_centers = pd.concat([data_centers, source_encoded], axis=1)

columns = [
    "power_source", "batteries", "biomass", "coal", "fuel_cells", 
    "geothermal", "grid_unspecified_mix", "hybrid_onsite_and_grid", 
    "hydroelectric", "natural_gas", "nuclear", "oil_diesel", 
    "renewable", "solar", "wind"
]

data_centers.loc[data_centers['power_source'].notna()][columns].sample(5)

,power_source,batteries,biomass,coal,fuel_cells,geothermal,grid_unspecified_mix,hybrid_onsite_and_grid,hydroelectric,natural_gas,nuclear,oil_diesel,renewable,solar,wind
39,Hydroelectric,0,0,0,0,0,0,0,1,0,0,0,0,0,0
511,Grid (unspecified mix),0,0,0,0,0,1,0,0,0,0,0,0,0,0
1140,Grid (unspecified mix),0,0,0,0,0,1,0,0,0,0,0,0,0,0
372,Natural gas,0,0,0,0,0,0,0,0,1,0,0,0,0,0
1084,Grid (unspecified mix),0,0,0,0,0,1,0,0,0,0,0,0,0,0


`number_of_buildings`

In [72]:
# Standardization function for assigning number of buildings
def standardize_buildings(buildings):
    if pd.isna(buildings):
        return np.nan, np.nan
    
    buildings = buildings.replace(' ', '')
    b_low = np.nan
    b_high = np.nan
    
    if '-' in buildings:
        split = buildings.split('-')
        b_low = int(split[0])
        b_high = int(split[1])
        
    elif '+' in buildings:
        buildings = buildings.replace('+', '')
        b_low = int(buildings)
        
    elif buildings.isdigit():
        b_low = int(buildings)
        b_high = int(buildings)
        
    return b_low, b_high
    
# Add new features to handle ranges
data_centers[["buildings_low", "buildings_high"]] = pd.DataFrame(
    data_centers["number_of_buildings"].apply(standardize_buildings).tolist(),
    index=data_centers.index
)

data_centers.loc[data_centers["buildings_low"].notna()][["number_of_buildings", "buildings_low", "buildings_high"]].head()

,number_of_buildings,buildings_low,buildings_high
2,18,18.0,18.0
10,4,4.0,4.0
11,2,2.0,2.0
14,14,14.0,14.0
15,6,6.0,6.0


`cooling_source` + `cooling_type`

In [73]:
for col in ['cooling_source', 'cooling_type']:
    # Normalize cooling cols
    cooling_data = (
        data_centers[col]
        .str.lower()
        .str.strip()
        .str.replace('/', '_')
        .str.replace(' ', '_')
    )

    # One-hot encoding for each source type
    source_encoded = cooling_data.str.get_dummies()
    data_centers = pd.concat([data_centers, source_encoded], axis=1)

columns = ['cooling_source', 'air', 'water', 'hybrid_air_water']

data_centers.loc[data_centers['cooling_source'].notna()][columns].sample(5)

,cooling_source,air,water,hybrid_air_water
221,Water,0,1,0
64,Hybrid air/water,0,0,1
464,Water,0,1,0
82,Water,0,1,0
1,Air,1,0,0


In [74]:
columns = ['cooling_type', 'closed_loop', 'fans', 'open_loop']

data_centers.loc[data_centers['cooling_type'].notna()][columns].sample(5)

,cooling_type,closed_loop,fans,open_loop
438,Closed loop,1,0,0
749,Closed loop,1,0,0
1020,Closed loop,1,0,0
462,Closed loop,1,0,0
902,Closed loop,1,0,0


`project_cost`

In [75]:
def standardize_project_cost(cost):
    if pd.isna(cost):
        return np.nan, np.nan
    
    multiplier = 1
    cost_low = np.nan
    cost_high = np.nan
    
    # Normalize cost
    cost = (
        str(cost)
        .lower()
        .replace(' ', '')
        .replace('$', '')
        .replace(',', '')
        .replace('miillion', 'million')
    )

    # Replace abbreviated m
    cost = re.sub(r'm$', 'million', cost)
    
    if 'million' in cost:
        multiplier = 1000000
    elif 'billion' in cost:
        multiplier = 1000000000
        
    cost = re.sub(r'[mb]illion', '', cost)
    
    # Assign range to low and high
    if '-' in cost:
        split = cost.split('-')
        cost_low = float(split[0]) * multiplier
        cost_high = float(split[1]) * multiplier
        
    elif '+' in cost:
        cost = cost.replace('+', '')
        cost_low = float(cost) * multiplier
        
    else:
        cost_low = float(cost) * multiplier
        cost_high = float(cost) * multiplier
        
    return cost_low, cost_high

# Add new features for handling the cost range
data_centers[["project_cost_low", "project_cost_high"]] = pd.DataFrame(
    data_centers["project_cost"].apply(standardize_project_cost).tolist(),
    index=data_centers.index
)

data_centers[data_centers["project_cost"].notna()][["project_cost", "project_cost_low", "project_cost_high"]].sample(5)

,project_cost,project_cost_low,project_cost_high
51,$3.6 billion,3.600000e+09,3.600000e+09
417,$1 billion,1.000000e+09,1.000000e+09
1068,14.5 billion,1.450000e+10,1.450000e+10
742,$6.2 billion,6.200000e+09,6.200000e+09
514,$1.2 billion,1.200000e+09,1.200000e+09


### FracTracker Crypto Mining Operations

In [76]:
crypto = pd.read_csv(DATA_DIR / "fractracker_crypto.csv")
print("Rows and columns:", crypto.shape)
crypto.sample(3)

Rows and columns: (302, 18)


,Name,City,State,Address,Company,Probable power source,Links,Status,lat,long,Precise Location?,MapCat,GEM_links,Other updates,Date,MW (NYTimes),Unnamed: 16,Unnamed: 17
170,Digipower X American Axle Erie County facility,Buffalo,NY,"1001 E Delavan Ave, Buffalo, NY 14215",Digipower X,Grid; Hydro,https://buffalonews.com/news/local/vacant-amer...,Operating,42.922280,-78.819630,Yes,Operating / under construction,https://www.gem.wiki/Digihost_American_Axle_Er...,NaN,4/17/2025,NaN,Digihost moving 28 MW of NYS holdings to Alaba...,NaN
207,Greenville facility,Greenville,SC,NaN,Genesis Digital Assets,Grid,https://www.goupstate.com/story/news/local/202...,Construction,34.845690,-82.402060,No,Operating / under construction,NaN,NaN,NaN,NaN,NaN,NaN
24,Juice Tech--Little Rock,Little Rock,AR,"3300 S. Woodrow St., Little Rock, AR",Juice Tech,NaN,https://www.arkansasbusiness.com/article/14121...,Proposed,34.720055,-92.306673,Yes,Proposed,NaN,NaN,7/27/2023,NaN,NaN,NaN


In [77]:
crypto.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Name,302,250,CleanSpark,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,283,235,Dalton,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
State,302,41,GA,38,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Address,194,189,"430 Davis Rd, Vidalia, GA 30474",4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Company,283,174,CleanSpark,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Probable power source,163,36,Unknown,54,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Links,299,263,https://www.wbir.com/article/tech/east-tenness...,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Status,301,11,Operating,138,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lat,302.0,NaN,NaN,NaN,37.830255,5.191345,25.454595,34.456008,36.356141,41.261212,69.71761
long,302.0,NaN,NaN,NaN,-128.525789,651.005258,-11402.57827,-98.202512,-88.774187,-82.477328,-69.170992


### FracTracker Crypto Mining Operations - Data Cleaning

| Feature | Identified Issue | Proposed Solution |
|---|---|---|
| `Probable power source` | Power source values use inconsistent naming conventions. | Normalize them to the FracTracker data center categories using keyword mapping. |
| `Precise Location?` | Values represented as 'Yes' / 'No' strings. | Update to a boolean representation. |
| `MW (NYTimes)` | A range of values including '100 MW', '100 MW+', '100 MW --> 200 MW'. | Standardize to `mw_low` and `mw_high` to keep consistent representation in both datasets. |

`Probable power source`

In [78]:
# Reuse power sources from FracTracker data centers
power_sources = [
    "batteries", "biomass", "coal", "fuel_cells",
    "geothermal", "grid_unspecified_mix", "hybrid_onsite_and_grid",
    "hydroelectric", "natural_gas", "nuclear", "oil_diesel",
    "renewable", "solar", "wind"
]

# Mapping given sources to predefined power sources
def normalize_sources(source):
    if pd.isna(source):
        return []

    source = source.lower()
    found_sources = set()

    # Keyword mappings
    keyword_map = {
        "battery": "batteries",
        "biomass": "biomass",
        "coal": "coal",
        "fuel cell": "fuel_cells",
        "geothermal": "geothermal",
        "hydro": "hydroelectric",
        "gas": "natural_gas",
        "nuclear": "nuclear",
        "diesel": "oil_diesel",
        "oil": "oil_diesel",
        "solar": "solar",
        "wind": "wind",
        "renewable": "renewable",
        "wte": "renewable",
    }

    for keyword, category in keyword_map.items():
        if keyword in source:
            found_sources.add(category)

    # Grid
    if "grid" in source:
        found_sources.add("grid_unspecified_mix")

    # Grid + another identified source = hybrid
    non_grid_sources = found_sources - {"grid_unspecified_mix"}

    if "grid_unspecified_mix" in found_sources and non_grid_sources:
        found_sources.add("hybrid_onsite_and_grid")

    return sorted(found_sources)


crypto["norm_power_source"] = crypto["Probable power source"].apply(normalize_sources)
crypto.loc[crypto["norm_power_source"].str.len() > 0][["norm_power_source"]].sample(5)

,norm_power_source
267,[natural_gas]
98,[grid_unspecified_mix]
162,"[grid_unspecified_mix, hybrid_onsite_and_grid,..."
281,"[grid_unspecified_mix, hybrid_onsite_and_grid,..."
194,[coal]


`Precise Location?`

In [79]:
# Change precise location to a boolean
crypto["Precise Location?"] = crypto["Precise Location?"].apply(lambda x: x == 'Yes')
crypto[['Precise Location?']].sample(5)

,Precise Location?
126,False
226,False
7,True
68,False
72,True


`MW (NYTimes)`

In [80]:
# Function for standardizing mw values
def standardize_mw(value):
    if pd.isna(value):
        return np.nan, np.nan

    text = str(value).lower().strip()

    # Fix typo
    text = text.replace("nw", "mw")

    # Extract numbers followed by MW or GW
    matches = re.findall(r'(\d+(?:\.\d+)?)\s*(mw|gw)', text)

    if not matches:
        return np.nan, np.nan

    values_mw = []

    for number, unit in matches:
        number = float(number)

        # Convert to MW
        if unit == "gw":
            number *= 1000

        values_mw.append(number)

    return min(values_mw), max(values_mw)


# Add new features to handle ranges
crypto[["mw_low", "mw_high"]] = pd.DataFrame(
    crypto["MW (NYTimes)"].apply(standardize_mw).tolist(),
    index=crypto.index
)

crypto.loc[crypto['MW (NYTimes)'].notna()][["MW (NYTimes)", "mw_low", "mw_high"]].sample(5)

,MW (NYTimes),mw_low,mw_high
258,Potential 1.4 GW solar?,1400.0,1400.0
202,120 MW,120.0,120.0
46,17.7 MW,17.7,17.7
219,60 MW,60.0,60.0
234,50+ MW,NaN,NaN


### Address Standardization & Comparison 

Addresses are standardized across both datasets to make facility matching more reliable. Common street names, directions, and unit types are converted to consistent abbreviations, while special characters and extra whitespace are removed. The crypto dataset sometimes stores the city, state, and ZIP code inside the Address field. These values are separated so the final location fields are: 

- **Crypto**: `Address`, `City`, `State`, `Zip` 
- **Data Centers**: `address`, `city`, `state`, `zip`

#### Address Abbreviations

In [81]:
ADDRESS_ABBREVIATIONS = {
    # Directions
    'north': 'n',
    'south': 's',
    'east': 'e',
    'west': 'w',
    'northeast': 'ne',
    'northwest': 'nw',
    'southeast': 'se',
    'southwest': 'sw',

    # Street suffixes
    'street': 'st',
    'avenue': 'ave',
    'road': 'rd',
    'drive': 'dr',
    'lane': 'ln',
    'boulevard': 'blvd',
    'circle': 'cir',
    'court': 'ct',
    'place': 'pl',
    'parkway': 'pkwy',
    'highway': 'hwy',
    'trail': 'trl',
    'terrace': 'ter',
    'square': 'sq',
    'crossing': 'xing',
    'center': 'ctr',
    'centre': 'ctr',
    'point': 'pt',
    'ridge': 'rdg',
    'run': 'run',
    'view': 'vw',
    'village': 'vlg',
    'station': 'sta',
    'landing': 'lndg',
    'junction': 'jct',
    'extension': 'ext',
    'expressway': 'expy',
    'freeway': 'fwy',
    'turnpike': 'tpke',
    'causeway': 'cswy',
    'crescent': 'cres',
    'cove': 'cv',
    'fork': 'frk',
    'garden': 'gdn',
    'gardens': 'gdns',
    'grove': 'grv',
    'harbor': 'hbr',
    'harbour': 'hbr',
    'heights': 'hts',
    'hill': 'hl',
    'hills': 'hls',
    'island': 'is',
    'islands': 'iss',
    'knoll': 'knl',
    'knolls': 'knls',
    'lake': 'lk',
    'lakes': 'lks',
    'manor': 'mnr',
    'manors': 'mnrs',
    'meadow': 'mdw',
    'meadows': 'mdws',
    'mount': 'mt',
    'mountain': 'mtn',
    'orchard': 'orch',
    'pine': 'pne',
    'pines': 'pnes',
    'plaza': 'plz',
    'port': 'prt',
    'prairie': 'pr',
    'radial': 'radl',
    'river': 'riv',
    'shore': 'shr',
    'shores': 'shrs',
    'spring': 'spg',
    'springs': 'spgs',
    'summit': 'smt',
    'union': 'un',
    'valley': 'vly',
    'valleys': 'vlys',
    'well': 'wl',
    'wells': 'wls',
    
    # Unit Overrides
    'apartment': 'apt',
    'apt.': 'apt',
    'suite': 'ste',
    'ste.': 'ste',
    'unit': 'unit',
    '#': 'unit',
    'building': 'bldg',
    'floor': 'fl',
    'room': 'rm',
    'department': 'dept'
}

#### Helper Functions

These functions are used to standardize addresses by removing special characters and extra whitespace and replacing common address terms with predefined abbreviations. In the crypto dataset, the `Address` field sometimes also contains the city, state, and ZIP code. A second function, `remove_city_state`, removes this duplicated location information, while ZIP codes are extracted into a separate column.

In [82]:
def standardize_address(address):
    if pd.isna(address):
        return np.nan
    
    address = address.lower().strip()
    
    # Remove special characters
    address = re.sub(r'[^a-z0-9\s]', '', address)
    address_parts = address.split()
    
    # Update addresses using abbreviations
    cleaned_parts = [
        ADDRESS_ABBREVIATIONS.get(part, part)
        for part in address_parts
    ]
    
    return " ".join(cleaned_parts)


def remove_city_state(address, city, state):
    if pd.isna(address):
        return np.nan

    address = str(address)

    # Remove ZIP from address
    address = re.sub(r'\s*,?\s*\d{5}\s*$', '', address)

    # Remove State from address
    if pd.notna(state):
        address = re.sub(
            rf'\s*,?\s*{re.escape(str(state))}\s*$',
            '',
            address,
            flags=re.IGNORECASE
        )

    # Remove City from address
    if pd.notna(city):
        address = re.sub(
            rf'\s*,?\s*{re.escape(str(city))}\s*$',
            '',
            address,
            flags=re.IGNORECASE
        )

    return address.strip(' ,')

In [83]:
# Extract 5-digit zip if exists
crypto['Zip'] = crypto['Address'].str.extract(r'(\d{5})\s*$')[0]
crypto['Zip'] = crypto['Zip'].replace('', np.nan).astype('Int64') # missing zip

# Remove city, state, and zip from addresses in crypto
crypto['Address'] = crypto.apply(
    lambda row: remove_city_state(
        row['Address'],
        row['City'],
        row['State']
    ),
    axis=1
)

# Applying the transformations to the addresses in both datasets
crypto['Address'] = crypto['Address'].apply(standardize_address)
data_centers['address'] = data_centers['address'].apply(standardize_address)


# data_centers[['address', 'city', 'state', 'zip']].sample(5)
crypto[['Address', 'City', 'State', 'Zip']].sample(5)

,Address,City,State,Zip
233,somewhere in w texas,NaN,TX,<NA>
41,325 2nd st,Cedartown,GA,30125
263,NaN,Odessa,TX,<NA>
15,241 main ave,Mountain Pine,AR,71956
34,1575 gardon of the gods rd,Colorado Springs,CO,<NA>


#### Address Comparison

The standardized datasets are merged using exact matches on address, city, and state to identify facilities that appear in both datasets. This allows us to identify data centers associated with known crypto mining locations, providing more information for later analysis. Using exact address matching, 19 matching records were identified between the two datasets.

In [84]:
# Find matching addresses between the datasets
matches = crypto[crypto['Address'].notna()].merge(
    data_centers[data_centers['address'].notna()],
    left_on=['Address', 'City', 'State'],
    right_on=['address', 'city', 'state'],
    how='inner',
    suffixes=('_crypto', '_dc')
)

print(f"{len(matches)} matches found between the crypto and data centers datatsets.")
matches[['facility_name', 'Name', 'lat_crypto', 'long_crypto', 'lat_dc', 'long_dc', 'address', 'city', 'state']]

19 matches found between the crypto and data centers datatsets.


,facility_name,Name,lat_crypto,long_crypto,lat_dc,long_dc,address,city,state
0,Core Scientific Dalton 1 & 2 Data Center Campus,Core Scientific Whitfield County facility (sel...,34.713160,-84.951430,34.713160,-84.951430,2205 industrial s rd,Dalton,GA
1,Riot 1,Block Mining McCracken County facility,37.088310,-88.703050,37.088310,-88.703050,5600 commerce dr,Paducah,KY
2,Riot 2,Riot Platforms greenfield site,37.088592,-88.676158,37.088593,-88.676155,coleman rd,Paducah,KY
3,Hyperscale Data Michigan Data Center,Michigan Data Center,41.993228,-86.098603,41.992455,-86.098976,415 e pr ronde st,Dowagiac,MI
4,CoreWeave/ Applied Digital,Applied Digital Stutsman County facility,47.027383,-98.679666,47.027080,-98.679600,2671 hwy 20 se,Jamestown,ND
5,Applied Digital: Polaris Forge 1,"Applied Digital, being repurposed from crypto ...",46.010438,-98.570800,46.012590,-98.570700,9685 87th ave se,Ellendale,ND
6,Vulcan Infrastructure and Power Inc (formerly ...,Greenidge Generation Yates County facility,42.676870,-76.944910,42.676870,-76.944910,590 plant rd,Dresden,NY
7,BlockFusion aka North East Data Center LLC,North East Data LLC Niagara facility,43.082864,-79.003495,43.083187,-79.003170,5380 frontier ave,Niagara Falls,NY
8,9Lives Data Center,Nine Lives Digital Group,43.123208,-79.040614,43.123890,-79.041180,3807 highland ave,Niagara Falls,NY
9,North Tonawanda Data Center,Digipower X Fortistar Niagara County facility,43.060860,-78.853760,43.060860,-78.853760,1070 erie ave,North Tonawanda,NY


### EIA-860 Plant Data

The first row of each EIA-860 Excel file is a title, so the column names are on the second row (`header=1`).

In [85]:
# Added header=1 to skip the first row

plants = pd.read_excel(DATA_DIR / "eia860_2025" / "2___Plant_Y2025.xlsx", header=1)
print("Rows and columns:", plants.shape)
plants.sample(3)

Rows and columns: (17349, 42)


,Utility ID,Utility Name,Plant Code,Plant Name,Street Address,City,State,Zip,County,Latitude,Longitude,NERC Region,Balancing Authority Code,Balancing Authority Name,Name of Water Source,Primary Purpose (NAICS Code),Regulatory Status,Sector,Sector Name,FERC Cogeneration Status,FERC Cogeneration Docket Number,FERC Small Power Producer Status,FERC Small Power Producer Docket Number,FERC Exempt Wholesale Generator Status,FERC Exempt Wholesale Generator Docket Number,Ash Impoundment?,Ash Impoundment Lined?,Ash Impoundment Status,Transmission or Distribution System Owner,Transmission or Distribution System Owner ID,Transmission or Distribution System Owner State,Grid Voltage (kV),Grid Voltage 2 (kV),Grid Voltage 3 (kV),Energy Storage,Natural Gas LDC Name,Natural Gas Pipeline Name 1,Natural Gas Pipeline Name 2,Natural Gas Pipeline Name 3,Pipeline Notes,Natural Gas Storage,Liquefied Natural Gas Storage
6256,57128,Bloom Energy,58302,Life Technologies Carlsbad,5781 Van Alley Way,Carlsbad,CA,92008,San Diego,33.137222,-117.286944,WECC,CISO,California Independent System Operator,NaN,22,NR,2,IPP Non-CHP,N,NaN,N,NaN,N,NaN,NaN,X,NaN,San Diego Gas & Electric Co,16609,CA,0.48,,,N,SOUTHERN CALIFORNIA GAS COMPANY,NaN,NaN,NaN,NaN,N,NaN
4683,56765,CER Colorado Bend Energy Partners LP,56350,Colorado Bend Energy Center,3821 South State Highway 60,Wharton,TX,77488,Wharton,29.2878,-96.0683,TRE,ERCO,"Electric Reliability Council of Texas, Inc.",Wells,22,NR,2,IPP Non-CHP,N,NaN,N,NaN,Y,07-42-000,N,X,NaN,CenterPoint Energy,8901,TX,138,,,N,NaN,ENERGY TRANSFER FUEL LP,GULF SOUTH PIPELINE COMPANY LP,NaN,NaN,N,X
684,18231,City of Stuart - (IA),1190,Stuart (IA),114 North Fremont Street,Stuart,IA,50250,Guthrie,41.503889,-94.3172,MRO,MISO,Midcontinent Independent Transmission System O...,Municipality,22,RE,1,Electric Utility,N,NaN,N,NaN,N,NaN,N,X,NaN,Central Iowa Power Cooperative,3258,NaN,69,,,N,MIDAMERICAN ENERGY COMPANY,MIDAMERICAN ENERGY COMPANY,NaN,NaN,NaN,N,X


In [86]:
plants.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Utility ID,17349.0,NaN,NaN,NaN,47106.087844,23529.704706,7.0,18642.0,60301.0,64545.0,68045.0
Utility Name,17349,6935,"Altus Power America Management, LLC",247,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Plant Code,17349.0,NaN,NaN,NaN,52099.079601,22868.185984,1.0,55701.0,61103.0,65921.0,70452.0
Plant Name,17349,17345,Viola,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Street Address,17229,15510,TBD,789,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,17314,6015,Lancaster,78,NaN,NaN,NaN,NaN,NaN,NaN,NaN
State,17349,51,CA,2250,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Zip,17349.0,8431.0,93501.0,64.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
County,17326,1542,Kern,304,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Latitude,17349,16666,,28,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### EIA-860 Generator Data

The generator file has three sheets: `Operable`, `Proposed`, and `Retired and Canceled`. For now we only look at `Operable`.

In [87]:
generators = pd.read_excel(
    DATA_DIR / "eia860_2025" / "3_1_Generator_Y2025.xlsx",
    sheet_name="Operable",
    header=1
)
print("Rows and columns:", generators.shape)
generators.sample(3)

Rows and columns: (27919, 73)


,Utility ID,Utility Name,Plant Code,Plant Name,State,County,Generator ID,Technology,Prime Mover,Unit Code,Ownership,Duct Burners,Can Bypass Heat Recovery Steam Generator?,RTO/ISO LMP Node Designation,RTO/ISO Location Designation for Reporting Wholesale Sales Data to FERC,Nameplate Capacity (MW),Nameplate Power Factor,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Uprate or Derate Completed During Year,Month Uprate or Derate Completed,Year Uprate or Derate Completed,Status,Synchronized to Transmission Grid,Operating Month,Operating Year,Planned Retirement Month,Planned Retirement Year,Associated with Combined Heat and Power System,Sector Name,Sector,Topping or Bottoming,Energy Source 1,Energy Source 2,Energy Source 3,Energy Source 4,Energy Source 5,Energy Source 6,Startup Source 1,Startup Source 2,Startup Source 3,Startup Source 4,Solid Fuel Gasification System?,Carbon Capture Technology?,Turbines or Hydrokinetic Buoys,Time from Cold Shutdown to Full Load,Fluidized Bed Technology?,Pulverized Coal Technology?,Stoker Technology?,Other Combustion Technology?,Subcritical Technology?,Supercritical Technology?,Ultrasupercritical Technology?,Planned Net Summer Capacity Uprate (MW),Planned Net Winter Capacity Uprate (MW),Planned Uprate Month,Planned Uprate Year,Planned Net Summer Capacity Derate (MW),Planned Net Winter Capacity Derate (MW),Planned Derate Month,Planned Derate Year,Planned New Prime Mover,Planned Energy Source 1,Planned New Nameplate Capacity (MW),Planned Repower Month,Planned Repower Year,Other Planned Modifications?,Other Modifications Month,Other Modifications Year,Multiple Fuels?,Cofire Fuels?,Switch Between Oil and Natural Gas?
12329,15142,Plains End Operating Services LLC,55650.0,Plains End,CO,Jefferson,GE16,Natural Gas Internal Combustion Engine,IC,NaN,S,X,X,NaN,NaN,5.7,0.8,5.52,5.52,3.6,N,,,OP,X,5.0,2002.0,,,N,IPP Non-CHP,2.0,X,NG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,N,,12H,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,,,,,,,,NaN,NaN,,,,N,,,N,N,N
7323,26616,North Slope Borough Power & Light (AK),7485.0,NSB Point Hope Utility,AK,North Slope,PG7,Petroleum Liquids,IC,NaN,S,X,X,NaN,NaN,0.6,0.8,0.6,0.6,0.2,N,,,OP,X,9.0,1995.0,,,N,Electric Utility,1.0,X,DFO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,N,,12H,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,,,,,,,,NaN,NaN,,,,N,,,N,N,NaN
7463,195,Alabama Power Co,7721.0,Theodore Cogen Facility,AL,Mobile,2,Natural Gas Fired Combined Cycle,CA,G525,S,Y,X,NaN,NaN,88.4,0.85,64.7,64.7,64.7,N,,,OP,X,12.0,2000.0,,,Y,Electric Utility,1.0,T,NG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,12H,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,,,,,,,,NaN,NaN,,,,NaN,,,N,NaN,NaN


In [88]:
generators.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Utility ID,27919.0,5853.0,55938.0,329.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Utility Name,27918,5842,Industrial Power Generating Company LLC,329,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Plant Code,27918.0,NaN,NaN,NaN,42375.107386,26782.287436,1.0,7146.0,56794.5,62643.75,70438.0
Plant Name,27918,14325,Edison Sault,73,NaN,NaN,NaN,NaN,NaN,NaN,NaN
State,27918,51,CA,3161,NaN,NaN,NaN,NaN,NaN,NaN,NaN
County,27918,1479,Los Angeles,424,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Generator ID,27917,10890,1,2923,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Technology,27918,27,Solar Photovoltaic,7999,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Prime Mover,27918,18,PV,7999,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Unit Code,2180,310,CC1,770,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### EIA-861M Retail Sales of Electricity


In [89]:
# Added header=2 to skip the first two rows of the Excel sheet

sales = pd.read_excel(DATA_DIR / "eia861m_sales_revenue.xlsx", sheet_name="Monthly-States", header=2)
print("Rows and columns:", sales.shape)
sales.sample(3)

Rows and columns: (10150, 24)


,Year,Month,State,Data Status,Thousand Dollars,Megawatthours,Count,Cents/kWh,Thousand Dollars.1,Megawatthours.1,Count.1,Cents/kWh.1,Thousand Dollars.2,Megawatthours.2,Count.2,Cents/kWh.2,Thousand Dollars.3,Megawatthours.3,Count.3,Cents/kWh.3,Thousand Dollars.4,Megawatthours.4,Count.4,Cents/kWh.4
6100,2016,8.0,NJ,Final,628743.08,3877079.4,3515597.0,16.22,504228.71,3866468.3,515468.0,13.04,73996.95,681836.31,11876.0,10.85,2434.51,26395.15,6.0,9.22,1209403.3,8451779.1,4042947.0,14.31
4905,2018,7.0,FL,Final,1499389.40,13123423.0,9409576.0,11.43,812172.79,9098991.7,1230356.0,8.93,117613.02,1539735.50,21667.0,7.64,615.03,7480.58,2.0,8.22,2429790.2,23769630.0,10661601.0,10.22
1896,2023,6.0,FL,Final,1912472.50,12554271.0,10214841.0,15.23,1026891.50,8668937.5,1293881.0,11.85,143227.23,1570419.20,26425.0,9.12,737.48,6545.55,2.0,11.27,3083328.8,22800173.0,11535149.0,13.52


In [90]:
sales.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Year,10150.0,18.0,2025.0,612.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Month,10149.0,NaN,NaN,NaN,6.41206,3.442597,1.0,3.0,6.0,9.0,12.0
State,10149,51,AK,199,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Data Status,10149,2,Final,9180,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Thousand Dollars,10149.0,NaN,NaN,NaN,322406.257117,374363.877149,13064.0,78291.42,218232.0,404119.32,3266301.5
Megawatthours,10149.0,NaN,NaN,NaN,2358136.574637,2557344.034097,104574.0,595182.0,1706973.5,3135968.7,22481421.0
Count,10149.0,NaN,NaN,NaN,2631403.392847,2731937.969772,226548.0,802281.0,1971280.0,3106777.0,14863374.0
Cents/kWh,10149.0,NaN,NaN,NaN,14.26151,5.25453,6.73,11.08,12.7,15.63,52.72
Thousand Dollars.1,10149.0,NaN,NaN,NaN,251960.525962,316264.755862,20778.0,66788.65,166221.67,313892.57,3327766.4
Megawatthours.1,10149.0,NaN,NaN,NaN,2241268.764867,2416984.780473,126343.74,642345.04,1623061.3,2826659.0,17843545.0


### EIA-930 Balance Data

In [91]:
# This is a subset of the EIA 930 Balance dataset for the second half of 2025

balance = pd.read_csv(DATA_DIR / "eia930" / "EIA930_BALANCE_2025_Jul_Dec.csv")
print("Rows and columns:", balance.shape)
balance.sample(3)

Rows and columns: (267218, 65)


,Balancing Authority,Data Date,Hour Number,Local Time at End of Hour,UTC Time at End of Hour,Demand Forecast (MW),Demand (MW),Net Generation (MW),Total Interchange (MW),Sum(Valid DIBAs) (MW),Demand (MW) (Imputed),Net Generation (MW) (Imputed),Total Interchange (MW) (Imputed),Demand (MW) (Adjusted),Net Generation (MW) (Adjusted),Total Interchange (MW) (Adjusted),Net Generation (MW) from Coal,Net Generation (MW) from Natural Gas,Net Generation (MW) from Nuclear,Net Generation (MW) from All Petroleum Products,Net Generation (MW) from Hydropower Excluding Pumped Storage,Net Generation (MW) from Pumped Storage,Net Generation (MW) from Solar without Integrated Battery Storage,Net Generation (MW) from Solar with Integrated Battery Storage,Net Generation (MW) from Wind without Integrated Battery Storage,Net Generation (MW) from Wind with Integrated Battery Storage,Net Generation (MW) from Battery Storage,Net Generation (MW) from Other Energy Storage,Net Generation (MW) from Unknown Energy Storage,Net Generation (MW) from Geothermal,Net Generation (MW) from Other Fuel Sources,Net Generation (MW) from Unknown Fuel Sources,Net Generation (MW) from Coal (Imputed),Net Generation (MW) from Natural Gas (Imputed),Net Generation (MW) from Nuclear (Imputed),Net Generation (MW) from All Petroleum Products (Imputed),Net Generation (MW) from Hydropower Excluding Pumped Storage (Imputed),Net Generation (MW) from Pumped Storage (Imputed),Net Generation (MW) from Solar without Integrated Battery Storage (Imputed),Net Generation (MW) from Solar with Integrated Battery Storage (Imputed),Net Generation (MW) from Wind without Integrated Battery Storage (Imputed),Net Generation (MW) from Wind with Integrated Battery Storage (Imputed),Net Generation (MW) from Battery Storage (Imputed),Net Generation (MW) from Other Energy Storage (Imputed),Net Generation (MW) from Unknown Energy Storage (Imputed),Net Generation (MW) from Geothermal (Imputed),Net Generation (MW) from Other Fuel Sources (Imputed),Net Generation (MW) from Unknown Fuel Sources (Imputed),Net Generation (MW) from Coal (Adjusted),Net Generation (MW) from Natural Gas (Adjusted),Net Generation (MW) from Nuclear (Adjusted),Net Generation (MW) from All Petroleum Products (Adjusted),Net Generation (MW) from Hydropower Excluding Pumped Storage (Adjusted),Net Generation (MW) from Pumped Storage (Adjusted),Net Generation (MW) from Solar without Integrated Battery Storage (Adjusted),Net Generation (MW) from Solar witho Integrated Battery Storage (Adjusted),Net Generation (MW) from Wind without Integrated Battery Storage (Adjusted),Net Generation (MW) from Wind with Integrated Battery Storage (Adjusted),Net Generation (MW) from Battery Storage (Adjusted),Net Generation (MW) from Other Energy Storage (Adjusted),Net Generation (MW) from Unknown Energy Storage (Adjusted),Net Generation (MW) from Geothermal (Adjusted),Net Generation (MW) from Other Fuel Sources (Adjusted),Net Generation (MW) from Unknown Fuel Sources (Adjusted),Region
201796,SIKE,11/04/2025,13,11/04/2025 1:00:00 PM,11/04/2025 7:00:00 PM,NaN,NaN,21.0,21.0,22.0,NaN,NaN,NaN,NaN,21.0,21.0,21.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MIDW
239136,TPWR,07/27/2025,3,07/27/2025 3:00:00 AM,07/27/2025 10:00:00 AM,331.0,321.0,73.0,-248.0,-248.0,NaN,NaN,NaN,321.0,73.0,-248.0,NaN,NaN,NaN,NaN,73.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,73.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NW
140793,NWMT,12/09/2025,7,12/09/2025 7:00:00 AM,12/09/2025 2:00:00 PM,1369.0,1362.0,2458.0,1096.0,1096.0,NaN,NaN,NaN,1362.0,2458.0,1096.0,743.0,197.0,NaN,58.0,371.0,NaN,0.0,NaN,1089.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,743.0,197.0,NaN,58.0,371.0,NaN,0.0,NaN,1089.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN

In [92]:
balance.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Balancing Authority,267218,61,AECI,4417,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Data Date,267218,184,11/02/2025,1490,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Hour Number,267218.0,NaN,NaN,NaN,12.502339,6.923663,1.0,7.0,13.0,19.0,25.0
Local Time at End of Hour,267218,4416,11/02/2025 1:00:00 AM,110,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UTC Time at End of Hour,267218,4420,07/01/2025 8:00:00 AM,61,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Demand Forecast (MW),233431.0,NaN,NaN,NaN,9465.17759,19303.64298,-227966.0,791.0,2449.0,6493.0,156738.0
Demand (MW),233639.0,NaN,NaN,NaN,9503.464246,19220.239208,-39388.0,815.0,2544.0,6665.0,810948.0
Net Generation (MW),266717.0,NaN,NaN,NaN,8304.861227,18525.972929,-2576.0,547.0,1759.0,5935.0,810705.0
Total Interchange (MW),266739.0,NaN,NaN,NaN,-78.283611,1344.438016,-11031.0,-496.0,-8.0,387.0,40651.0
Sum(Valid DIBAs) (MW),265658.0,NaN,NaN,NaN,-60.813783,1416.714082,-57015.0,-505.0,-13.0,374.0,40651.0


### EIA-930 Subregion Data

In [93]:
subregion = pd.read_csv(DATA_DIR / "eia930" / "EIA930_SUBREGION_2025_Jul_Dec.csv")
print("Rows and columns:", subregion.shape)
subregion.sample(3)

Rows and columns: (366596, 7)


,Balancing Authority,Data Date,Hour Number,Sub-Region,Demand (MW),Local Time at End of Hour,UTC Time at End of Hour
348069,SWPP,11/16/2025,15,INDN,87.0,11/16/2025 3:00:00 PM,11/16/2025 9:00:00 PM
133034,NYIS,09/07/2025,23,ZONE,716.0,09/07/2025 11:00:00 PM,09/08/2025 3:00:00 AM
307996,SWPP,08/10/2025,10,WR,3654.0,08/10/2025 10:00:00 AM,08/10/2025 3:00:00 PM


In [94]:
subregion.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Balancing Authority,366596,8,PJM,88340,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Data Date,366596,184,11/02/2025,2060,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Hour Number,366596.0,NaN,NaN,NaN,12.502319,6.923647,1.0,7.0,13.0,19.0,25.0
Sub-Region,366596,83,PGAE,4417,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Demand (MW),365441.0,NaN,NaN,NaN,3942.233748,4968.20665,-4.0,814.0,1889.0,4780.0,32436.0
Local Time at End of Hour,366596,4416,11/02/2025 1:00:00 AM,151,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UTC Time at End of Hour,366596,4420,07/01/2025 8:00:00 AM,83,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Save cleaned data

Save the cleaned data centers to `data/processed/` so later notebooks can load them without re-running this one.

In [95]:
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

data_centers.to_csv(PROCESSED_DIR / "fractracker_data_centers_clean.csv", index=False)
print("Saved: fractracker_data_centers_clean.csv with", data_centers.shape[0], "rows and", data_centers.shape[1], "columns")

Saved: fractracker_data_centers_clean.csv with 1701 rows and 77 columns
